# 1_3. Survival ML: catboost

In [9]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
import optuna
import matplotlib.pyplot as plt
from optuna_integration import CatBoostPruningCallback

## Deciduous

In [10]:
deciduous_teeth = pd.read_csv("../data/deciduous_teeth.csv", sep=";")

### Preprocess

In order to use right-censored data in catboost use -1 to express inf

In [11]:
deciduous_teeth.loc[((deciduous_teeth["LOWER"].isna()), "LOWER")] = 0
deciduous_teeth.loc[((deciduous_teeth["UPPER"].isna()), "UPPER")] = -1

In [12]:
deciduous_teeth.columns

Index(['DOG', 'TOOTH_NUMBER', 'MAX_MAND', 'SIDE', 'POSITION', 'DEVELOPM_STAGE',
       'LOWER', 'UPPER', 'CENS', 'SEX', 'BREED', 'BREED_SIZE', 'SKULL_TYPE',
       'BREED_CLADE'],
      dtype='object')

In [13]:
deciduous_teeth = deciduous_teeth.drop(columns=["BREED"])

In [14]:
features = deciduous_teeth.columns.difference(
    ["LOWER", "UPPER", "DOG", "CENS"], sort=False
)

# catboost categorical features can't be nan
deciduous_teeth = deciduous_teeth.fillna("unknown")

cat_features = [
    "MAX_MAND",
    "SIDE",
    "SEX",
    "BREED_SIZE",
    "SKULL_TYPE",
    "BREED_CLADE",
]
num_features = features.difference(cat_features, sort=False)

### Hyperparam search

In [15]:
features

Index(['TOOTH_NUMBER', 'MAX_MAND', 'SIDE', 'POSITION', 'DEVELOPM_STAGE', 'SEX',
       'BREED_SIZE', 'SKULL_TYPE', 'BREED_CLADE'],
      dtype='object')

In [ ]:
# Define hyperparameter search space
base_params = {
    "verbose": 0,
    "eval_metric": "SurvivalAft",
    "random_seed": 42,
    "monotone_constraints": {"DEVELOPM_STAGE": 1},
    "boosting_type": "Ordered",
}  # Hyperparameters common to all trials
feature_combos = [
    features,  # alle features
    [
        "DEVELOPM_STAGE",
        "MAX_MAND",
        "SIDE",
        "POSITION",
        "SEX",
        "BREED_SIZE",
        "SKULL_TYPE",
        "BREED_CLADE",
    ],
    [
        "DEVELOPM_STAGE",
        "TOOTH_NUMBER",
        "SEX",
        "BREED_SIZE",
        "SKULL_TYPE",
        "BREED_CLADE",
    ],
    [
        "TOOTH_NUMBER",
        "MAX_MAND",
        "SIDE",
        "POSITION",
        "DEVELOPM_STAGE",
        "BREED_SIZE",
        "SKULL_TYPE",
    ],
    [
        "DEVELOPM_STAGE",
        "TOOTH_NUMBER",
        "BREED_SIZE",
        "SKULL_TYPE",
    ],
]

N_OUTER_FOLDS = 5
N_INNER_FOLDS = 3
test_preds = {}

lodo = StratifiedGroupKFold(N_OUTER_FOLDS, shuffle=True, random_state=42)
for i, (train_index, test_index) in enumerate(
    lodo.split(
        X=deciduous_teeth, y=deciduous_teeth["CENS"], groups=deciduous_teeth["DOG"]
    )
):
    print(i)
    train_outer = deciduous_teeth.iloc[train_index]
    test_outer = deciduous_teeth.iloc[test_index]

    def objective(trial):
        try:
            # Define hyperparameter search space
            feature_selection = trial.suggest_int(
                "feature_selection", 0, len(feature_combos) - 1
            )

            distribution = trial.suggest_categorical(
                "dist", ["Normal", "Extreme", "Logistic"]
            )
            scale = trial.suggest_float("scale", 0.01, 5, log=True)

            loss_function = f"SurvivalAft:dist={distribution};scale={scale:.6f}"
            params = {
                "depth": trial.suggest_int("depth", 2, 8),
                "learning_rate": trial.suggest_float(
                    "learning_rate", 0.001, 0.3, log=True
                ),
                "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.5, 1.0),
                "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 10),
                "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
                "bagging_temperature": trial.suggest_float(
                    "bagging_temperature", 0, 1.0
                ),
                "random_strength": trial.suggest_float("random_strength", 0, 1),
                "loss_function": loss_function,
            }
            params.update(base_params)
            pruning_callback = CatBoostPruningCallback(trial, "SurvivalAft")

            # CV
            train_events = deciduous_teeth.loc[train_index, "CENS"]
            train_ids = deciduous_teeth.loc[train_index, "DOG"]
            best_iters = []  # to save best amount of iterations of inner loop
            sgkf = StratifiedGroupKFold(
                n_splits=N_INNER_FOLDS, shuffle=True, random_state=42
            )
            CV_scores = []
            CV_features = feature_combos[feature_selection]
            CV_cat_features = list(set(cat_features).intersection(CV_features))

            for idx, (train_idx_inner, test_idx_inner) in enumerate(
                sgkf.split(train_outer, train_events, train_ids)
            ):
                train_inner, test_inner = (
                    train_outer.iloc[train_idx_inner],
                    train_outer.iloc[test_idx_inner],
                )

                train_pool_CV = Pool(
                    train_inner[CV_features],
                    label=train_inner.loc[:, ["LOWER", "UPPER"]],
                    cat_features=CV_cat_features,
                    group_id=train_inner["DOG"],
                )
                valid_pool_CV = Pool(
                    test_inner[CV_features],
                    label=test_inner.loc[:, ["LOWER", "UPPER"]],
                    cat_features=CV_cat_features,
                    group_id=test_inner["DOG"],
                )

                model = CatBoostRegressor(**params)

                if idx == 0:
                    model.fit(
                        train_pool_CV,
                        eval_set=valid_pool_CV,
                        early_stopping_rounds=100,
                        verbose=0,
                        callbacks=[pruning_callback],
                    )
                else:
                    model.fit(
                        train_pool_CV,
                        eval_set=valid_pool_CV,
                        early_stopping_rounds=100,
                        verbose=0,
                    )

                best_iters.append(model.get_best_iteration())
                CV_scores.append(model.get_best_score()["validation"]["SurvivalAft"])

            trial.set_user_attr("best_iteration", int(np.median(best_iters)))
            return np.mean(CV_scores)
        except Exception as e:
            print(f"Skipping trial due to error: {e}")
            return float("inf")

    # Optuna for hyperparam tuning
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=50, show_progress_bar=False)

    # get best params out of inner loop
    best_iter = study.best_trial.user_attrs["best_iteration"]
    best_params = {}
    best_params.update(base_params)
    best_params.update(study.best_trial.params)
    best_features = best_params.pop("feature_selection")
    best_dist = best_params.pop("dist")
    best_scale = best_params.pop("scale")
    best_params["loss_function"] = (
        f"SurvivalAft:dist={best_dist};scale={best_scale:.6f}"
    )
    best_cat_features = list(
        set(cat_features).intersection(feature_combos[best_features])
    )

    # pool maken
    train_outer_pool = Pool(
        train_outer[feature_combos[best_features]],
        label=train_outer.loc[:, ["LOWER", "UPPER"]],
        cat_features=best_cat_features,
        group_id=train_outer["DOG"],
    )
    test_outer_pool = Pool(
        test_outer[feature_combos[best_features]],
        label=test_outer.loc[:, ["LOWER", "UPPER"]],
        cat_features=best_cat_features,
        group_id=test_outer["DOG"],
    )

    bst_outer = CatBoostRegressor(**best_params, iterations=best_iter)
    bst_outer.fit(train_outer_pool, verbose=0)
    preds_outer = bst_outer.predict(test_outer_pool, prediction_type="RawFormulaVal")
    # Store by original indices
    for idx, pred_outer in zip(test_index, preds_outer):
        test_preds[idx] = {
            "pred": pred_outer,
            "dist": best_dist,
            "scale": best_scale,
        }

In [ ]:
# Convert to DataFrame in original order
test_preds_df = pd.DataFrame.from_dict(test_preds, orient="index").sort_index()

In [ ]:
test_preds_df.to_csv(r"preds/cb_tooth_preds_dec.csv", sep=";")

## Permanent

In [17]:
permanent_teeth = pd.read_csv("../data/permanent_teeth.csv", sep=";")

### Preprocess

In [18]:
permanent_teeth.loc[((permanent_teeth["LOWER"].isna()), "LOWER")] = 0
permanent_teeth.loc[((permanent_teeth["UPPER"].isna()), "UPPER")] = -1

In [19]:
permanent_teeth = permanent_teeth.drop(columns=["BREED"])

In [20]:
features = permanent_teeth.columns.difference(
    ["LOWER", "UPPER", "DOG", "CENS"], sort=False
)

# catboost categorical features can't be nan
permanent_teeth = permanent_teeth.fillna("unknown")

cat_features = [
    "MAX_MAND",
    "SIDE",
    "SEX",
    "BREED_SIZE",
    "SKULL_TYPE",
    "BREED_CLADE",
]
num_features = features.difference(cat_features, sort=False)

In [21]:
features

Index(['TOOTH_NUMBER', 'MAX_MAND', 'SIDE', 'POSITION', 'DEVELOPM_STAGE', 'SEX',
       'BREED_SIZE', 'SKULL_TYPE', 'BREED_CLADE'],
      dtype='object')

### Hyperparam search

In [ ]:
# Define hyperparameter search space
base_params = {
    "verbose": 0,
    "eval_metric": "SurvivalAft",
    "random_seed": 42,
    "boosting_type": "Ordered",
}  # Hyperparameters common to all trials
feature_combos = [
    features,  # alle features
    [
        "DEVELOPM_STAGE",
        "MAX_MAND",
        "SIDE",
        "POSITION",
        "SEX",
        "BREED_SIZE",
        "SKULL_TYPE",
        "BREED_CLADE",
    ],
    [
        "DEVELOPM_STAGE",
        "TOOTH_NUMBER",
        "SEX",
        "BREED_SIZE",
        "SKULL_TYPE",
        "BREED_CLADE",
    ],
    [
        "TOOTH_NUMBER",
        "MAX_MAND",
        "SIDE",
        "POSITION",
        "DEVELOPM_STAGE",
        "BREED_SIZE",
        "SKULL_TYPE",
    ],
    [
        "DEVELOPM_STAGE",
        "TOOTH_NUMBER",
        "BREED_SIZE",
        "SKULL_TYPE",
    ],
]

N_OUTER_FOLDS = 5
N_INNER_FOLDS = 3
test_preds = {}

lodo = StratifiedGroupKFold(N_OUTER_FOLDS, shuffle=True, random_state=42)
for i, (train_index, test_index) in enumerate(
    lodo.split(
        X=permanent_teeth, y=permanent_teeth["CENS"], groups=permanent_teeth["DOG"]
    )
):
    print(i)
    train_outer = permanent_teeth.iloc[train_index]
    test_outer = permanent_teeth.iloc[test_index]

    def objective(trial):
        try:
            # Define hyperparameter search space
            feature_selection = trial.suggest_int(
                "feature_selection", 0, len(feature_combos) - 1
            )

            distribution = trial.suggest_categorical(
                "dist", ["Normal", "Extreme", "Logistic"]
            )
            scale = trial.suggest_float("scale", 0.01, 5, log=True)

            loss_function = f"SurvivalAft:dist={distribution};scale={scale:.6f}"
            params = {
                "depth": trial.suggest_int("depth", 2, 8),
                "learning_rate": trial.suggest_float(
                    "learning_rate", 0.001, 0.3, log=True
                ),
                "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.5, 1.0),
                "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 10),
                "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
                "bagging_temperature": trial.suggest_float(
                    "bagging_temperature", 0, 1.0
                ),
                "random_strength": trial.suggest_float("random_strength", 0, 1),
                "loss_function": loss_function,
            }
            params.update(base_params)
            pruning_callback = CatBoostPruningCallback(trial, "SurvivalAft")

            # CV
            train_events = permanent_teeth.loc[train_index, "CENS"]
            train_ids = permanent_teeth.loc[train_index, "DOG"]
            best_iters = []  # to save best amount of iterations of inner loop
            sgkf = StratifiedGroupKFold(
                n_splits=N_INNER_FOLDS, shuffle=True, random_state=42
            )
            CV_scores = []
            CV_features = feature_combos[feature_selection]
            CV_cat_features = list(set(cat_features).intersection(CV_features))

            for idx, (train_idx_inner, test_idx_inner) in enumerate(
                sgkf.split(train_outer, train_events, train_ids)
            ):
                train_inner, test_inner = (
                    train_outer.iloc[train_idx_inner],
                    train_outer.iloc[test_idx_inner],
                )

                train_pool_CV = Pool(
                    train_inner[CV_features],
                    label=train_inner.loc[:, ["LOWER", "UPPER"]],
                    cat_features=CV_cat_features,
                    group_id=train_inner["DOG"],
                )
                valid_pool_CV = Pool(
                    test_inner[CV_features],
                    label=test_inner.loc[:, ["LOWER", "UPPER"]],
                    cat_features=CV_cat_features,
                    group_id=test_inner["DOG"],
                )

                model = CatBoostRegressor(**params)

                if idx == 0:
                    model.fit(
                        train_pool_CV,
                        eval_set=valid_pool_CV,
                        early_stopping_rounds=100,
                        verbose=0,
                        callbacks=[pruning_callback],
                    )
                else:
                    model.fit(
                        train_pool_CV,
                        eval_set=valid_pool_CV,
                        early_stopping_rounds=100,
                        verbose=0,
                    )

                best_iters.append(model.get_best_iteration())
                CV_scores.append(model.get_best_score()["validation"]["SurvivalAft"])

            trial.set_user_attr("best_iteration", int(np.median(best_iters)))
            return np.mean(CV_scores)
        except Exception as e:
            print(f"Skipping trial due to error: {e}")
            return float("inf")

    # Optuna for hyperparam tuning
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=50, show_progress_bar=False)

    # get best params out of inner loop
    best_iter = study.best_trial.user_attrs["best_iteration"]
    best_params = {}
    best_params.update(base_params)
    best_params.update(study.best_trial.params)
    best_features = best_params.pop("feature_selection")
    best_dist = best_params.pop("dist")
    best_scale = best_params.pop("scale")
    best_params["loss_function"] = (
        f"SurvivalAft:dist={best_dist};scale={best_scale:.6f}"
    )
    best_cat_features = list(
        set(cat_features).intersection(feature_combos[best_features])
    )

    # pool maken
    train_outer_pool = Pool(
        train_outer[feature_combos[best_features]],
        label=train_outer.loc[:, ["LOWER", "UPPER"]],
        cat_features=best_cat_features,
        group_id=train_outer["DOG"],
    )
    test_outer_pool = Pool(
        test_outer[feature_combos[best_features]],
        label=test_outer.loc[:, ["LOWER", "UPPER"]],
        cat_features=best_cat_features,
        group_id=test_outer["DOG"],
    )

    bst_outer = CatBoostRegressor(**best_params, iterations=best_iter)
    bst_outer.fit(train_outer_pool, verbose=0)
    preds_outer = bst_outer.predict(test_outer_pool, prediction_type="RawFormulaVal")
    # Store by original indices
    for idx, pred_outer in zip(test_index, preds_outer):
        test_preds[idx] = {
            "pred": pred_outer,
            "dist": best_dist,
            "scale": best_scale,
        }

In [ ]:
# Convert to DataFrame in original order
test_preds_df = pd.DataFrame.from_dict(test_preds, orient="index").sort_index()

In [ ]:
test_preds_df.to_csv(r"preds/cb_tooth_preds_perma.csv", sep=";")